<a href="https://colab.research.google.com/github/irfanizzath/Personal-Notebooks/blob/main/Explainable_AI_Notes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Explainable AI in Python**

---

# Module 1: Foundations of Explainable AI

## 1.1 Introduction to Explainable AI

**What is Explainable AI?**
Explainable AI (XAI) refers to the set of methods, techniques, and tools that make the behavior and predictions of machine learning models understandable to humans. As models grow more complex (deep neural networks, ensembles, gradient boosting machines), their internal decision logic becomes opaque — often called a "black box." XAI aims to open that box, or at least approximate what's inside it.

**Why does explainability matter?**
- **Trust** — Stakeholders (doctors, loan officers, judges) need to trust automated decisions before acting on them.
- **Debugging** — Explanations reveal when a model is relying on spurious correlations (e.g., a "husky vs wolf" classifier keying on snow in the background).
- **Regulatory compliance** — Laws like GDPR (EU) include a "right to explanation" for automated decisions affecting individuals; sectors like finance and healthcare often mandate interpretability.
- **Fairness and bias detection** — Explanations help surface whether protected attributes (race, gender, age) are implicitly driving predictions.
- **Model improvement** — Understanding *why* a model errs helps engineers fix data quality or feature engineering issues.

**Key vocabulary**
| Term | Meaning |
|---|---|
| Interpretability | The degree to which a human can understand the cause of a decision, intrinsically (by design) |
| Explainability | Techniques applied *after* or *around* a model to make its behavior understandable, even if the model itself isn't inherently interpretable |
| Transparency | The model's mechanics are visible/inspectable (e.g., a decision tree's splits) |
| Post-hoc explanation | An explanation generated after training, without altering the model, often by probing input-output behavior |
| Global explanation | Describes overall model behavior across the entire dataset |
| Local explanation | Describes why the model made one specific prediction for one specific instance |

**The interpretability–performance trade-off**
Historically, simpler models (linear regression, decision trees) are easier to interpret but may sacrifice predictive accuracy on complex data. Complex models (deep nets, gradient boosted trees, ensembles) often perform better but are harder to interpret. XAI techniques try to bridge this gap — letting practitioners use powerful models while still recovering interpretable insights.

---

## 1.2 Decision Trees vs. Neural Networks

| Aspect | Decision Trees | Neural Networks |
|---|---|---|
| Structure | Hierarchical if-then-else rules based on feature thresholds | Layers of weighted connections and nonlinear activations |
| Interpretability | High — you can trace a path from root to leaf | Low — millions of weights interacting nonlinearly |
| Explanation method | Native (structure IS the explanation) | Requires post-hoc methods (SHAP, LIME, saliency maps) |
| Decision logic | Explicit, rule-based, easy to convert to natural language | Implicit, distributed across weights, not directly readable |
| Typical use in XAI courses | Used as the "ground truth" example of an inherently interpretable model | Used as the canonical example requiring model-agnostic explainability tools |

**Key takeaway:** Decision trees are *intrinsically interpretable* (interpretability by design), while neural networks require *extrinsic/post-hoc* interpretability techniques. This distinction sets up the rest of the course: some models explain themselves; others need external tools.

---

## 1.3 Model-Agnostic vs. Model-Specific Explainability

**Model-specific explainability**
- Techniques that exploit knowledge of a *particular* model's internal structure.
- Examples: coefficients in linear regression, Gini/entropy-based feature importance in decision trees, attention weights in transformers.
- Pros: Computationally cheap, exact (not approximated), tightly tied to how the model actually computes its output.
- Cons: Not portable — a technique built for linear models cannot be applied to a random forest or neural network.

**Model-agnostic explainability**
- Techniques that treat the model as a black box: they only need access to inputs and outputs (predictions or probabilities), not internal parameters.
- Examples: Permutation Importance, SHAP, LIME, Partial Dependence Plots.
- Pros: Works with *any* model — decision trees, SVMs, neural nets, ensembles — and enables apples-to-apples comparison across model types.
- Cons: Often more computationally expensive (many perturbations/samples needed); results are approximations, not exact internal calculations.

**Rule of thumb:** Use model-specific methods when available and appropriate (they're faster and exact); fall back to model-agnostic methods when comparing across model types, using complex/ensemble/deep models, or when the model type may change in production.

---

## 1.4 Explainability in Linear Models

Linear and logistic regression are the most naturally interpretable models because their prediction function is a simple weighted sum:

```
ŷ = β0 + β1x1 + β2x2 + ... + βn xn
```

Each coefficient βi directly quantifies how much the prediction changes when feature xi increases by one unit, holding all else constant.

### 1.4.1 Computing Feature Impact with Linear Regression

- **Coefficient magnitude and sign** directly indicate feature impact: positive β → feature increases prediction; negative β → feature decreases prediction.
- **Caveat — feature scale matters.** Raw coefficients are not comparable across features unless features are standardized (zero mean, unit variance). A coefficient of 1000 for "income in dollars" is not "more important" than 0.5 for "age in years" without standardization.
- **Standardized coefficients**: Fit the model on z-scored features so all coefficients are on the same scale, making magnitude comparisons meaningful.

```python
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)
model = LinearRegression().fit(X_scaled, y)

feature_impact = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
```

- **Confidence intervals / p-values** (via `statsmodels`) tell you whether a coefficient's effect is statistically significant, not just its magnitude.

### 1.4.2 Computing Feature Impact with Logistic Regression

- Logistic regression models the *log-odds* of the positive class:
```
log(p / (1-p)) = β0 + β1x1 + ... + βn xn
```
- Coefficients represent the change in **log-odds** per unit increase in the feature — not a direct probability change (since the sigmoid is nonlinear).
- To make coefficients more interpretable, exponentiate them: `exp(βi)` gives the **odds ratio** — how much the odds of the positive outcome multiply for a one-unit increase in xi.
  - `exp(βi) > 1` → feature increases odds of positive class.
  - `exp(βi) < 1` → feature decreases odds.
  - `exp(βi) = 1` → no effect.

```python
from sklearn.linear_model import LogisticRegression

model = LogisticRegression().fit(X_scaled, y)
odds_ratios = pd.Series(np.exp(model.coef_[0]), index=X.columns)
```

---

## 1.5 Explainability in Tree-Based Models

Trees split the feature space using thresholds that maximize "purity" (reduction in Gini impurity or entropy for classification, variance reduction for regression). Because splits are explicit, we can extract exact decision rules and compute feature importance natively.

### 1.5.1 Computing Feature Importance with Decision Trees

**Extracting decision rules:** Every path from root to leaf is a conjunction of conditions (e.g., `IF age > 45 AND income <= 50000 THEN class = 1`). Scikit-learn allows this extraction directly:

```python
from sklearn.tree import export_text, DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=4).fit(X, y)
print(export_text(tree, feature_names=list(X.columns)))
```

**Feature importance (Gini importance / Mean Decrease in Impurity — MDI):**
- For each feature, sum the impurity decrease it produces across all nodes where it's used as a splitter, weighted by the fraction of samples reaching that node.
- Normalized so importances sum to 1.

```python
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
```

- **Limitation:** MDI is biased toward high-cardinality features (features with many unique values get more chances to be selected as splitters), which can inflate their apparent importance.

### 1.5.2 Computing Feature Importance with Random Forests

- Random forests average feature importances across all trees in the ensemble, which reduces variance and gives a more stable estimate than a single tree.
- Still inherits the same MDI bias toward high-cardinality/continuous features.
- **Alternative:** Permutation importance (covered in Module 2) fixes this bias by measuring importance via actual predictive degradation rather than impurity reduction internal to training.

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X, y)
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
```

**Visualizing feature importance** (common to both trees and forests):
```python
import matplotlib.pyplot as plt
importance.plot(kind='barh')
plt.xlabel("Feature Importance")
plt.title("Feature Importance (Random Forest)")
plt.show()
```

---
---

# Module 2: Model-Agnostic Explainability

## 2.1 Permutation Importance

**Concept:** Measures how much a model's performance (accuracy, MSE, etc.) drops when a single feature's values are randomly shuffled (permuted), breaking its relationship with the target while keeping the marginal distribution of that feature intact.

**Algorithm:**
1. Train the model and record baseline performance on a hold-out set.
2. For each feature, shuffle its column, keep all others fixed, and re-predict.
3. Compute the performance drop = baseline score − permuted score.
4. Larger drop → more important feature (the model relied on it heavily).
5. Repeat multiple times (different random permutations) and average, to reduce noise.

```python
from sklearn.inspection import permutation_importance

result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42, scoring='accuracy')
importances = pd.Series(result.importances_mean, index=X.columns).sort_values(ascending=False)
```

**Advantages over MDI:**
- Model-agnostic — works for any fitted estimator.
- Unbiased with respect to feature cardinality.
- Reflects actual predictive contribution on unseen (test) data, not just training-time impurity reduction.

**Caveats:**
- Computationally more expensive (requires re-scoring many times).
- Can give misleading results when features are highly correlated (shuffling one correlated feature may not hurt performance much, since a correlated partner still carries similar information).

### 2.1.1 Permutation Importance for MLPClassifier

Neural networks (e.g., scikit-learn's `MLPClassifier`) have no native `.feature_importances_` attribute, so permutation importance is one of the few practical ways to interpret them.

```python
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500).fit(X_train, y_train)
result = permutation_importance(mlp, X_test, y_test, n_repeats=20, random_state=1)
```

This demonstrates the power of model-agnostic tools: the exact same function call works whether the underlying model is a random forest or a neural network.

### 2.1.2 Coefficients vs. Permutation Importance

| | Linear Coefficients | Permutation Importance |
|---|---|---|
| Applicability | Linear models only | Any model |
| What it measures | Marginal effect of feature holding others constant (model-internal) | Drop in predictive performance when feature signal is destroyed |
| Sensitive to correlated features? | Yes — multicollinearity distorts coefficient estimates | Yes — but for a different reason (redundant info from correlated features) |
| Requires standardized features for fair comparison? | Yes | Less critical (measured via performance drop, not raw magnitude) |
| Computation cost | Free (already estimated during fit) | Expensive (requires repeated re-scoring) |

**Practical insight taught in this lesson:** On a linear model, comparing standardized coefficients against permutation importance rankings is a good sanity check — a large mismatch may indicate multicollinearity or nonlinearity that the linear model isn't capturing well.

---

## 2.2 SHAP Explainability

**SHAP (SHapley Additive exPlanations)** is grounded in cooperative game theory. It treats each feature as a "player" contributing to a "payout" (the model's prediction), and computes each feature's fair share of credit using **Shapley values**.

**Core idea:** For a given prediction, consider all possible subsets ("coalitions") of features. The Shapley value for feature *i* is the average marginal contribution of adding feature *i* across all possible orderings of features being added to the coalition.

```
φi = Σ (over subsets S not containing i)  [|S|!(n-|S|-1)! / n!] × [f(S ∪ {i}) − f(S)]
```

**Key properties (why SHAP is theoretically attractive):**
- **Local accuracy (efficiency):** The sum of all feature SHAP values plus the baseline (expected prediction) equals the actual model output for that instance.
- **Consistency:** If a model changes so that a feature's contribution increases (or stays the same) regardless of other features, that feature's SHAP value never decreases.
- **Missingness:** Features that don't affect the prediction get a SHAP value of zero.

**Interpretation:** SHAP values can be computed **globally** (average absolute SHAP value per feature across all instances = overall importance) or **locally** (SHAP values for one instance explain that individual prediction).

### 2.2.1 Finding Key Medical Charge Predictors with SHAP
Example use case: predicting medical insurance charges (regression) from features like age, BMI, smoker status, number of children, region. SHAP typically reveals `smoker` as the dominant driver, with `age` and `BMI` as secondary contributors — and crucially quantifies *how much* each contributes in dollar terms per individual.

```python
import shap
explainer = shap.TreeExplainer(model)  # for tree-based models
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)
```

### 2.2.2 Finding Key Heart Disease Predictors with SHAP
Example use case: classification of heart disease risk. SHAP applied to a classifier (e.g., random forest or gradient boosting) surfaces clinically meaningful drivers (e.g., chest pain type, max heart rate, ST depression) and — importantly — shows *directionality* (whether high or low values push toward "disease" vs. "no disease").

---

## 2.3 SHAP Kernel Explainer

**KernelSHAP** is the model-agnostic implementation of SHAP. Unlike `TreeExplainer` (exact, fast, tree-specific) or `LinearExplainer` (exact for linear models), `KernelExplainer` works with *any* black-box model by:
1. Sampling coalitions of features (some "present," some replaced with background/reference values).
2. Getting model predictions for each coalition.
3. Fitting a weighted linear regression (using the Shapley kernel weighting) to approximate each feature's Shapley value.

**Trade-off:** Model-agnostic and universally applicable, but computationally expensive and approximate (relies on sampling), especially with many features.

```python
background = shap.sample(X_train, 100)  # background/reference dataset
explainer = shap.KernelExplainer(model.predict, background)
shap_values = explainer.shap_values(X_test.iloc[:50])
```

### 2.3.1 Kernel Explainer for MLPRegressor
Applied to a neural network regressor with no tree structure to exploit — KernelExplainer treats the MLP purely as a prediction function, sampling perturbed inputs and observing output changes.

### 2.3.2 Kernel Explainer for MLPClassifier
Same principle applied to classification — KernelExplainer uses `predict_proba` as the function being explained, producing SHAP values per class.

```python
explainer = shap.KernelExplainer(mlp.predict_proba, background)
shap_values = explainer.shap_values(X_test.iloc[:50])
```

---

## 2.4 SHAP vs. Model-Specific Approaches

| | Model-Specific SHAP (Tree/Linear Explainer) | KernelSHAP (model-agnostic) |
|---|---|---|
| Speed | Fast (polynomial-time exact algorithms exploiting model structure) | Slow (sampling-based approximation) |
| Accuracy | Exact Shapley values | Approximate, sample-size dependent |
| Applicability | Requires matching explainer to model type | Works with any model |
| When to use | Whenever a matching fast explainer exists (tree ensembles, linear/logistic models, some deep learning explainers like DeepExplainer) | Only when no faster specialized explainer exists (e.g., arbitrary black-box pipelines, MLPs, SVMs) |

**Practical guidance:** Always prefer a model-specific SHAP explainer when one exists for your model family; reserve KernelExplainer for genuinely black-box or heterogeneous pipeline scenarios.

---

## 2.5 Visualizing SHAP Explainability

### 2.5.1 Feature Importance Plots (e.g., Admissions Analysis)
A **bar plot** of mean absolute SHAP values ranks features by their overall (global) contribution to predictions — e.g., in a graduate admissions dataset, GPA and standardized test scores typically dominate over less predictive features like extracurricular count.

```python
shap.summary_plot(shap_values, X_test, plot_type="bar")
```

### 2.5.2 Beeswarm Plots — Analyzing Feature Effects
The **beeswarm plot** is SHAP's signature visualization: each dot is one instance-feature pair. Position on the x-axis = SHAP value (impact on prediction); color = feature value (red = high, blue = low); features are stacked vertically by importance.

```python
shap.summary_plot(shap_values, X_test)  # default = beeswarm
```

**Reading a beeswarm plot:** If red dots cluster on the positive side and blue dots on the negative side for a feature, high values of that feature *increase* the prediction (and vice-versa for the opposite pattern). This simultaneously conveys importance ranking, direction of effect, and the distribution of effects across the dataset — something a single bar chart cannot show.

### 2.5.3 Partial Dependence Plots (PDPs) — Assessing Impact
**PDP** shows how the *average* predicted outcome changes as one feature varies across its range, marginalizing out all other features.

**Algorithm:** For a grid of values of feature xi, replace that feature's value for *every* instance in the dataset with each grid value, get predictions, and average them. Plot average prediction vs. feature value.

```python
from sklearn.inspection import PartialDependenceDisplay
PartialDependenceDisplay.from_estimator(model, X_train, features=['age', 'bmi'])
```

**PDP vs. SHAP dependence plots:**
- PDP shows an *average* global trend for a feature — good for understanding overall marginal effect but can mask heterogeneous effects (e.g., if the feature helps one subgroup and hurts another, they may average out to a flat line).
- SHAP dependence plots show the *actual* per-instance relationship (each point is a real SHAP value for a real instance), revealing heterogeneity and interaction effects (often shown via color-coding a second, interacting feature).

---
---

# Module 3: Local Explainability

## 3.1 Global vs. Local Explainability

| | Global Explainability | Local Explainability |
|---|---|---|
| Question answered | "What drives the model overall, across the whole dataset?" | "Why did the model predict *this* for *this specific instance*?" |
| Techniques | Feature importance (MDI, permutation), summary/beeswarm SHAP plots, PDPs | SHAP force/waterfall plots for a single instance, LIME |
| Use case | Model auditing, feature selection, understanding general behavior | Explaining an individual decision to an end user (e.g., "why was this loan denied?") |

## 3.2 Local Explainability with SHAP

Because SHAP values are additive per-instance (they sum to the model's output for that instance minus the baseline expectation), they are naturally suited to local explanation. Two common visualizations:

- **Force plot:** Shows features pushing the prediction higher (red) or lower (blue) from the baseline, for one instance.
- **Waterfall plot:** A sequential, top-to-bottom bar chart showing how each feature's SHAP value moves the prediction from the baseline (expected value) to the final output.

```python
shap.plots.waterfall(shap_values[0])  # explain a single prediction (instance index 0)
shap.plots.force(explainer.expected_value, shap_values[0], X_test.iloc[0])
```

### 3.2.1 SHAP for Explaining Income Levels
Example use case: a binary classifier predicting whether an individual's income exceeds $50K (e.g., UCI Adult/Census dataset). Local SHAP explanations reveal, for a specific person, exactly which factors (education level, hours worked, occupation, capital gain) pushed their predicted income classification up or down — useful for auditing fairness on a per-individual basis (e.g., checking whether protected attributes like sex or race disproportionately drive predictions for certain individuals).

---

## 3.3 Local Explainability with LIME

**LIME (Local Interpretable Model-agnostic Explanations)** explains an individual prediction by approximating the black-box model *locally* with a simple, interpretable surrogate model (usually sparse linear regression).

**Algorithm:**
1. Pick the instance to explain.
2. Generate perturbed samples around that instance (small random variations in feature values).
3. Get the black-box model's predictions for each perturbed sample.
4. Weight the perturbed samples by proximity to the original instance (closer = higher weight).
5. Fit a simple, interpretable model (e.g., weighted linear regression, or a shallow decision tree) on this local neighborhood.
6. The coefficients/rules of this local surrogate model are the explanation — valid only in the vicinity of that specific instance.

**Key distinction from SHAP:** LIME approximates local behavior with a *separate, simple model fit on perturbed data*; SHAP computes *exact or theoretically-grounded* attribution values based on cooperative game theory. LIME is generally faster and more intuitive but has weaker theoretical guarantees (results can vary with perturbation sampling and kernel width choice).

### 3.3.1 Interpreting Regressors Locally
```python
import lime
import lime.lime_tabular

explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, feature_names=X_train.columns, mode='regression'
)
exp = explainer.explain_instance(X_test.iloc[0].values, model.predict, num_features=8)
exp.show_in_notebook()
```
For a regression task (e.g., predicting house price or medical charges), LIME shows which features pushed the prediction above/below the local average, with approximate magnitudes.

### 3.3.2 Interpreting Classifiers Locally
```python
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train.values, feature_names=X_train.columns,
    class_names=['no', 'yes'], mode='classification'
)
exp = explainer.explain_instance(X_test.iloc[0].values, model.predict_proba, num_features=8)
```
For classification, LIME shows which feature-value ranges (e.g., "age > 45", "glucose <= 120") contributed toward or against the predicted class, along with each factor's approximate weight.

---

## 3.4 Text and Image Explainability with LIME

LIME's perturbation strategy generalizes beyond tabular data by defining domain-appropriate ways to perturb inputs:
- **Text:** perturb by randomly removing/masking words from the input sentence.
- **Images:** perturb by dividing the image into "superpixels" (contiguous segments) and randomly turning segments on/off (e.g., graying them out).

### 3.4.1 Explaining Sentiment Analysis Predictions (Text)
```python
from lime.lime_text import LimeTextExplainer

explainer = LimeTextExplainer(class_names=['negative', 'positive'])
exp = explainer.explain_instance(review_text, sentiment_model.predict_proba, num_features=10)
exp.show_in_notebook()
```
**How it works:** LIME creates many variants of the input review with random words removed, observes how the sentiment prediction shifts, and highlights which specific words (e.g., "terrible," "loved," "boring") most strongly drive the prediction toward positive or negative sentiment. This produces intuitive, highlighted-text explanations.

### 3.4.2 Explaining Food Image Predictions (Image)
```python
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer = lime_image.LimeImageExplainer()
explanation = explainer.explain_instance(image_array, classifier_fn=model.predict, top_labels=5, num_samples=1000)
temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=True, num_features=5)
plt.imshow(mark_boundaries(temp, mask))
```
**How it works:** The image is segmented into superpixels; LIME perturbs by masking out random subsets of superpixels and observes how classification confidence (e.g., "pizza" vs. "sushi") changes. The output highlights the specific image regions (e.g., the cheese and crust of a pizza) that most influenced the model's prediction — visually intuitive for verifying the model is "looking at" the right things rather than background artifacts.

---
---

# Module 4: Advanced Topics in Explainable AI

## 4.1 Explainability Metrics

Since explanation methods themselves are estimators/approximations, we need ways to evaluate whether an explanation is *good* — i.e., trustworthy and meaningful, not just plausible-looking. Two key properties are commonly assessed: **consistency** and **faithfulness**.

### 4.1.1 Evaluating SHAP Explanation Consistency
**Consistency** asks: do repeated runs of the explanation method (or small, semantically irrelevant changes to the setup) produce stable, similar explanations? For SHAP specifically:
- Since KernelSHAP is sampling-based, running it multiple times with different random seeds/background samples should yield similar (not wildly different) SHAP values if the explanation is reliable.
- **Method:** Compute SHAP values multiple times, measure variance or rank correlation (e.g., Spearman correlation) of feature importance rankings across runs. Low variance / high correlation = consistent explanations.

```python
from scipy.stats import spearmanr
corr, _ = spearmanr(shap_run1_mean_abs, shap_run2_mean_abs)
```

### 4.1.2 Evaluating Faithfulness with LIME
**Faithfulness** asks: does the explanation actually reflect how the underlying black-box model behaves, or is it a plausible-looking fiction? A common test:
1. Take the top-k features LIME identifies as most important for a prediction.
2. **Perturb/remove** those features (e.g., set to zero, mean-impute, or mask) and re-run the *actual black-box model* (not the surrogate).
3. If the explanation is faithful, removing the "important" features should cause a **large change** in the true model's prediction; removing "unimportant" features should cause little change.
4. This can be quantified via metrics like **comprehensiveness** (prediction drop when removing top features) and **sufficiency** (prediction retained when *only* top features are kept).

**Why this matters:** LIME's surrogate model is fit on a local neighborhood and can sometimes fail to capture the true model's behavior faithfully, especially with a poorly chosen kernel width or non-linear local decision boundaries — faithfulness testing is how you catch that.

---

## 4.2 Explaining Unsupervised Models

Explainability is harder for unsupervised learning (e.g., clustering) since there's no ground-truth label to attribute predictions to — instead, we must explain *cluster assignments* or *cluster structure*.

### 4.2.1 Feature Impact on Cluster Quality
**Approach:** Systematically remove or permute one feature at a time, re-run the clustering algorithm (e.g., k-means), and measure how much cluster *quality* (e.g., silhouette score, inertia) changes. A large quality drop when a feature is removed indicates that feature is important for defining meaningful cluster structure.

```python
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans

baseline_score = silhouette_score(X, KMeans(n_clusters=k).fit_predict(X))
for col in X.columns:
    X_dropped = X.drop(columns=[col])
    score = silhouette_score(X_dropped, KMeans(n_clusters=k).fit_predict(X_dropped))
    print(col, baseline_score - score)
```

### 4.2.2 Feature Importance in Clustering with ARI (Adjusted Rand Index)
**ARI** measures the similarity between two clustering assignments, correcting for chance agreement (range: -1 to 1, where 1 = identical clustering).

**Approach for feature importance:**
1. Run clustering on the full feature set → get baseline cluster labels.
2. Remove (or permute) one feature, re-run clustering → get new cluster labels.
3. Compute ARI between baseline labels and new labels.
4. **Low ARI** (labels changed a lot) → that feature was important in determining cluster structure.
5. **High ARI** (labels barely changed) → that feature had little influence on the clustering outcome.

```python
from sklearn.metrics import adjusted_rand_score

baseline_labels = KMeans(n_clusters=k, random_state=0).fit_predict(X)
for col in X.columns:
    X_dropped = X.drop(columns=[col])
    new_labels = KMeans(n_clusters=k, random_state=0).fit_predict(X_dropped)
    ari = adjusted_rand_score(baseline_labels, new_labels)
    print(f"{col}: ARI={ari:.3f} (importance ≈ {1-ari:.3f})")
```

This gives an interpretable, quantitative ranking of which features drive the cluster structure — analogous in spirit to permutation importance, but adapted for the unsupervised setting.

---

## 4.3 Explaining Chat-Based Generative AI Models

Large language models (LLMs) like ChatGPT present a different explainability challenge: there's no fixed feature vector or tabular structure — the "input" is a prompt (text) and the "output" is generated text, often produced through multi-step reasoning. Two prominent techniques for probing LLM reasoning:

### 4.3.1 Chain-of-Thought (CoT) to Discover Reasoning
**Chain-of-thought prompting** asks the model to articulate its intermediate reasoning steps before giving a final answer (e.g., "Let's think step by step..."), rather than jumping straight to a conclusion.

**As an explainability tool:**
- CoT output serves as a (partial) window into the model's "reasoning process" — you can inspect *which* facts, sub-calculations, or logical steps the model used to arrive at its answer.
- It allows humans to catch reasoning errors even when the final answer looks plausible, and to verify whether the stated reasoning actually supports the conclusion (sometimes it doesn't — a phenomenon sometimes called *unfaithful* chain-of-thought, where the stated reasoning doesn't fully reflect what actually drove the output).
- **Caveat:** CoT explanations are themselves generated text from the same model — they are a *reported* rationalization, not a verified causal trace of internal computation (unlike SHAP or permutation importance, which measure actual input-output sensitivity). CoT should be treated as a useful but imperfect self-report.

### 4.3.2 Self-Consistency to Assess Confidence
**Self-consistency** is a technique to improve and evaluate reliability of chain-of-thought reasoning:
1. Sample multiple independent chain-of-thought completions for the same prompt (using temperature > 0 for diversity).
2. Extract the final answer from each sampled reasoning path.
3. Take a majority vote across the different final answers as the consensus answer.

**As an explainability/confidence signal:**
- If most sampled reasoning paths converge on the same answer, that suggests higher model confidence/robustness in that answer.
- If answers vary widely across samples, that signals the model is uncertain, the question is ambiguous, or the reasoning process is unstable — a useful diagnostic for knowing when to trust (or double-check) a generated answer.
- This connects back to the *consistency* metric idea from Section 4.1 — just as we check whether SHAP/LIME explanations are stable across runs, self-consistency checks whether an LLM's reasoning/answer is stable across sampled generations.

```python
# Conceptual pseudocode
answers = []
for _ in range(10):
    response = llm.generate(prompt, temperature=0.7)
    answers.append(extract_final_answer(response))

from collections import Counter
consensus_answer, vote_count = Counter(answers).most_common(1)[0]
confidence = vote_count / len(answers)
```

---

# Course Summary / Key Takeaways

1. **Interpretability is a spectrum** — from intrinsically transparent models (linear regression, decision trees) to fully opaque ones (deep neural nets) requiring post-hoc tools.
2. **Model-specific vs. model-agnostic** is a foundational lens: use fast, exact model-specific methods when possible; use flexible, model-agnostic methods (permutation importance, SHAP, LIME) when comparing across model types or explaining black boxes.
3. **Global explanations** (feature importance, beeswarm plots, PDPs) describe overall model behavior; **local explanations** (SHAP waterfall/force plots, LIME) describe individual predictions — both are necessary for a complete picture.
4. **SHAP** offers theoretically grounded, consistent attributions (game-theoretic Shapley values); **LIME** offers fast, intuitive local surrogate-model explanations — each with distinct trade-offs in speed, exactness, and applicability (including to text and image data).
5. **Explanations need their own evaluation** — consistency (stability across runs) and faithfulness (do explanations reflect true model behavior) are essential checks before trusting any XAI output.
6. **Explainability extends beyond supervised tabular models** — to unsupervised clustering (via feature-removal + ARI/silhouette impact) and to generative/LLM systems (via chain-of-thought and self-consistency), showing that XAI is a general discipline applicable across the modern AI landscape.